# Germinal Antibody Design

![Germinal Antibody Design](https://proto-bio.github.io/proto-assets/images/tool/germinal/hero.png)

De novo epitope-targeted antibody design (VHH or scFv) using the Germinal pipeline ([Mille-Fragoso et al. 2026](https://www.nature.com/articles/s41587-026-03187-0)). This wrapper installs the upstream [`SantiagoMille/germinal`](https://github.com/SantiagoMille/germinal) repo at a pinned commit and runs one end-to-end campaign per call.

The pipeline is: **ColabDesign + AF2-Multimer hallucination → AbMPNN sequence redesign → Chai-1 / AF3 / Protenix structure validation → multi-stage filter cascade**.

> **Note: long-running tool.** Binder design campaigns take a long time on GPU, so the design cells below are left unexecuted and shown for reference.

In [ ]:
from pathlib import Path

from proto_tools.tools.binder_design import (
    GerminalConfig,
    GerminalInput,
    run_germinal_design,
)
from proto_tools.utils.notebook_docs import display_api_reference


## Input API Reference

In [ ]:
display_api_reference("germinal", "input")

## Config API Reference

In [ ]:
display_api_reference("germinal", "config")

## Output API Reference

In [ ]:
display_api_reference("germinal", "output")

In [ ]:
# Resolve the in-tree PD-L1 test target by walking up to the repo root.
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pdb_path = _repo_root / "tests" / "dummy_data" / "pdl1.pdb"
print(f"Using target PDB: {pdb_path}")

inputs = GerminalInput(
    target_pdb=str(pdb_path),
    target_chain="A",
    binder_chain="B",
    hotspots=["A56", "A66", "A115"],
    target_name="pdl1",
)

config = GerminalConfig(
    design_type="vhh",
    max_trajectories=2,
    max_passing_designs=1,
    structure_model="chai",
    germinal_overrides={
        "logits_steps": 20,
        "softmax_steps": 4,
        "search_steps": 1,
        "num_seqs": 2,
    },
)

result = run_germinal_design(inputs, config)
print(f"success={result.success}")
print(f"num_designs={result.num_designs}, num_accepted={result.num_accepted}")
print(f"pipeline_stats={result.pipeline_stats}")

In [ ]:
for design in result.designs[:3]:
    print(f"--- {design.design_id} ({design.stage_passed}) ---")
    print(f"sequence_heavy: {design.sequence_heavy}")
    if design.sequence_light:
        print(f"sequence_light: {design.sequence_light}")
    if 'i_ptm' in design.metrics:
        print(f"  i_ptm={design.metrics.i_ptm:.3f}")
    if 'plddt' in design.metrics:
        print(f"  plddt={design.metrics.plddt:.3f}")

# Export every produced design's PDB (one file per design_id under <export_path>/<name.lower()>/)
if result.designs:
    result.export(name="pdl1_designs", export_path="./outputs", file_format="pdb")
    print(f"Exported {result.num_designs} designs ({result.num_accepted} accepted) to ./outputs/pdl1_designs/")

In [ ]:
# Advanced configuration: scFv mode + custom filter override + verbatim Hydra settings.
# This cell only constructs the config (no GPU / no subprocess) so the notebook
# can be executed end-to-end on hosts without GPU.
advanced_config = GerminalConfig(
    design_type="scfv",
    max_trajectories=10,
    max_passing_designs=2,
    structure_model="chai",
    iptm_threshold=0.80,  # Tighten the iptm filter beyond Germinal's preset (0.74)
    filter_overrides={
        "final": {
            # Require at least 4 H-bonds at the interface
            "interface_hbonds": {"value": 4, "operator": ">="},
        },
    },
    germinal_overrides={
        # Emphasise interface pTM in the hallucination loss
        "weights_iptm": 1.0,
        # Disable the helix bias for floppy / loopy CDR designs
        "weights_helix": 0.0,
    },
)

print("Advanced scFv config:")
print(f"  design_type        = {advanced_config.design_type}")
print(f"  structure_model    = {advanced_config.structure_model}")
print(f"  iptm_threshold     = {advanced_config.iptm_threshold}")
print(f"  filter_overrides   = {advanced_config.filter_overrides}")
print(f"  germinal_overrides = {advanced_config.germinal_overrides}")
# To actually run: result = run_germinal_design(inputs, advanced_config)